# ICS 591 Team 2 — Stage 2a: Building the local (Ghana) waste image dataset

**Goal:** collect and label locally relevant waste images to fine-tune the selected **ViT-S/16** model.

**Pipeline:**
1. **Search** image engines with Ghana-specific keywords (local brands, places, slang).
2. **Download** the images, keeping each image's source URL for attribution.
3. **Clean**: remove broken, tiny, exact-duplicate and near-duplicate images.
4. **Pre-screen** with CLIP, a zero-shot model that *suggests* a label and flags junk such as logos, posters and portraits.
5. **Label** by hand in a click-through tool, split across the team, with a shared overlap set to measure agreement.
6. **Export** a clean `dataset/<class>/` folder plus a manifest for the fine-tuning notebook.

Everything saves to Google Drive after each step, so you can stop and resume at any point.

## Classes (what the classifier will learn)

| Label | Meaning | System decision |
|---|---|---|
| `plastic_clean` | Mainly plastic (bottles, sachets, containers, bags), sorted or reasonably clean | **accept** |
| `plastic_contaminated` | Plastic visibly mixed with food, liquid, soil or other rubbish (typical *borla* heap containing plastic) | **reject: contaminated** |
| `non_plastic` | Mainly another material: cans or scrap metal, glass, paper or cardboard, food or organics, textiles | **reject: wrong type** |
| *discard* | Not usable: no waste visible, poster, logo, map, text, cartoon, collage, heavy watermark, or a clearly identifiable person | not used |

## Labelling rules (agree on these before starting)
- Label what **dominates the image**, not what the search keyword was. Keywords only decide where we look.
- If you hesitate for more than about 3 seconds between two classes, pick the **more conservative** one (`plastic_contaminated` over `plastic_clean`). If it's genuinely unreadable, discard it.
- **Discard any image where a person's face is clearly identifiable** (Act 843 and basic ethics). A small, distant figure is fine.
- Don't fix the CLIP suggestion just because it's there. It is a hint, often wrong.

## Usage and ethics
Images are collected for **non-commercial academic research only**. We keep the source URL of every image, do not redistribute the images or publish the dataset, and share only the trained model and aggregate results.

**How to run:** Runtime → T4 GPU. Run cells 1–7 once (one person). Then **each teammate** runs cells 1–2 and the labelling cell 8 with their own name and shard number.

In [ ]:
# 1. Setup
!pip -q install ddgs icrawler imagehash open_clip_torch
import os, re, io, json, time, random, hashlib, pathlib, csv
import numpy as np, pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = 50_000_000

from google.colab import drive
drive.mount('/content/drive')
OUT = pathlib.Path('/content/drive/MyDrive/ics591_local_data')
RAW = OUT / 'raw'; RAW.mkdir(parents=True, exist_ok=True)
print('Working folder:', OUT)

CLASSES = ['plastic_clean', 'plastic_contaminated', 'non_plastic']
DISCARD = ['discard_junk', 'discard_person']

In [ ]:
# 2. Search keywords (edit freely)
# The bucket name only records WHY we searched; the real label is assigned by a human in step 8.
# Local terms used: 'pure water' = sachet water, 'rubber' = plastic bags, 'borla' = rubbish,
# 'aboboyaa' = waste tricycle, Zoomlion = major waste company; Voltic, Bel-Aqua, Verna = water brands.
QUERIES = {
    'plastic': [
        'pure water sachets waste Ghana', 'empty pure water sachets Accra', 'sachet water rubbish Ghana',
        'plastic bottles collected recycling Ghana', 'empty Voltic bottles', 'Bel-Aqua bottle waste',
        'Verna water bottle waste', 'PET bottles recycling Accra', 'plastic recycling Kumasi',
        'plastic waste bales Ghana', 'plastic bottles sack Ghana recycling', 'Coliba recycling Ghana plastic',
        'Nelplast Ghana plastic waste', 'black rubber bags waste Ghana', 'plastic containers waste Accra',
        'plastic waste collectors Ghana', 'sorted plastic waste Africa', 'plastic bottle waste Nigeria Ghana',
    ],
    'mixed_household': [
        'borla Accra', 'household waste Accra', 'rubbish bin overflowing Accra', 'waste bin Madina Accra',
        'refuse dump Madina', 'Madina market waste', 'Zoomlion waste bin Ghana', 'aboboyaa rubbish collection',
        'aboboyaa waste Accra', 'Accra gutter plastic waste', 'Odaw river plastic waste', 'Korle lagoon plastic',
        'Kpone landfill waste', 'Kumasi refuse dump', 'domestic waste Ghana', 'waste sack Ghana household',
        'plastic food waste mixed Ghana', 'rubbish heap Accra street',
    ],
    'other_material': [
        'scrap metal cans Ghana', 'Agbogbloshie scrap metal', 'empty tins waste Ghana', 'used milk tins Ghana',
        'glass bottles waste Ghana', 'beer bottles waste Accra', 'Kantamanto textile waste',
        'second hand clothes waste Accra', 'food waste market Accra', 'coconut husk waste Accra',
        'plantain peels waste Ghana', 'cardboard waste Ghana market', 'paper waste Ghana', 'e-waste Ghana',
    ],
}
PER_QUERY = 100     # images requested per query (the engines often return fewer)
print(sum(len(v) for v in QUERIES.values()), 'queries')

In [ ]:
# 3. Search (resumable). Builds raw/search_results.csv with image URL + source page for attribution.
from ddgs import DDGS
RESULTS = RAW / 'search_results.csv'
done = set(pd.read_csv(RESULTS)['query']) if RESULTS.exists() else set()
rows = []

def search(q):
    for attempt in range(4):
        try:
            return DDGS().images(q, region='wt-wt', safesearch='moderate', max_results=PER_QUERY, type_image='photo')
        except Exception as e:
            wait = 15 * (attempt + 1)
            print(f'   rate-limited/error ({type(e).__name__}); waiting {wait}s')
            time.sleep(wait)
    return []

for bucket, qs in QUERIES.items():
    for q in qs:
        if q in done: continue
        res = search(q)
        print(f'{len(res):4d}  {q}')
        new = [{'bucket': bucket, 'query': q, 'image_url': r.get('image'), 'source_page': r.get('url'),
                'title': r.get('title'), 'width': r.get('width'), 'height': r.get('height')} for r in res]
        if not new:   # record the query anyway so a rerun doesn't retry forever
            new = [{'bucket': bucket, 'query': q, 'image_url': None}]
        pd.DataFrame(new).to_csv(RESULTS, mode='a', header=not RESULTS.exists(), index=False)
        time.sleep(random.uniform(2, 4))

sr = pd.read_csv(RESULTS).dropna(subset=['image_url']).drop_duplicates('image_url')
print('\nUnique image URLs found:', len(sr))
sr.groupby('bucket').size()

In [ ]:
# 3b. OPTIONAL fallback: only run if step 3 returned very few results (e.g. search engine blocking Colab).
# Uses Bing via icrawler; source URLs are still recorded.
RUN_FALLBACK = False
if RUN_FALLBACK:
    from icrawler import ImageDownloader
    from icrawler.builtin import BingImageCrawler
    FB_LOG = RAW / 'bing_fallback_sources.csv'
    for bucket, qs in QUERIES.items():
        for q in qs:
            folder = RAW / 'bing' / re.sub(r'\W+', '_', q)
            class Rec(ImageDownloader):
                def get_filename(self, task, default_ext, _q=q, _b=bucket, _f=folder):
                    name = super().get_filename(task, default_ext)
                    with open(FB_LOG, 'a', newline='') as f:
                        csv.writer(f).writerow([_b, _q, task.get('file_url'), str(_f / name)])
                    return name
            BingImageCrawler(downloader_cls=Rec, downloader_threads=4,
                             storage={'root_dir': str(folder)}).crawl(keyword=q, max_num=PER_QUERY)
    print('Fallback done; images in', RAW / 'bing')

In [ ]:
# 4. Download (resumable, parallel). Keeps only real photos with short side >= 200 px.
import requests
from concurrent.futures import ThreadPoolExecutor
IMG_DIR = RAW / 'images'; IMG_DIR.mkdir(exist_ok=True)
DL_LOG = RAW / 'downloads.csv'
HEADERS = {'User-Agent': 'Mozilla/5.0 (academic research; ICS591 waste-classification project)'}
MIN_SIDE = 200

already = set(pd.read_csv(DL_LOG)['image_url']) if DL_LOG.exists() else set()
todo = sr[~sr['image_url'].isin(already)].to_dict('records')
print(len(todo), 'to download')

def fetch(r):
    out = {'image_url': r['image_url'], 'status': 'fail', 'file': None, 'sha1': None}
    try:
        resp = requests.get(r['image_url'], headers=HEADERS, timeout=15)
        resp.raise_for_status()
        im = Image.open(io.BytesIO(resp.content)); im.load()
        if im.mode in ('P', 'LA', 'RGBA'): im = im.convert('RGBA').convert('RGB')
        else: im = im.convert('RGB')
        if min(im.size) < MIN_SIDE:
            out['status'] = 'too_small'; return out
        s = 768 / max(im.size)
        if s < 1: im = im.resize((round(im.width * s), round(im.height * s)), Image.BICUBIC)
        buf = io.BytesIO(); im.save(buf, 'JPEG', quality=92); data = buf.getvalue()
        sha = hashlib.sha1(data).hexdigest()
        path = IMG_DIR / f'{sha}.jpg'
        if not path.exists(): path.write_bytes(data)
        out.update(status='ok', file=path.name, sha1=sha)
    except Exception as e:
        out['status'] = 'fail:' + type(e).__name__
    return out

BATCH = 200
for i in range(0, len(todo), BATCH):
    with ThreadPoolExecutor(16) as ex:
        res = list(ex.map(fetch, todo[i:i + BATCH]))
    pd.DataFrame(res).to_csv(DL_LOG, mode='a', header=not DL_LOG.exists(), index=False)
    print(f'{min(i + BATCH, len(todo))}/{len(todo)} processed')

dl = pd.read_csv(DL_LOG)
print(dl['status'].str.split(':').str[0].value_counts())

In [ ]:
# 5. Remove near-duplicates (the same photo is often reposted, resized or cropped across many sites)
import imagehash
ok = dl[dl['status'] == 'ok'].merge(sr, on='image_url', how='left').drop_duplicates('file')
print('Unique files after exact dedupe:', len(ok))

def phash_bits(f):
    try: return imagehash.phash(Image.open(IMG_DIR / f)).hash.flatten()
    except Exception: return None

with ThreadPoolExecutor(8) as ex:
    bits = list(ex.map(phash_bits, ok['file']))
ok['bits'] = bits; ok = ok[ok['bits'].notna()].reset_index(drop=True)
ok['pixels'] = [np.prod(Image.open(IMG_DIR / f).size) for f in ok['file']]
ok = ok.sort_values('pixels', ascending=False).reset_index(drop=True)   # keep the largest copy

HAMMING = 6   # <= 6 of 64 bits different => treat as the same photo
kept_bits, keep, dup_of = [], [], {}
for i, b in enumerate(ok['bits']):
    if kept_bits:
        d = (np.array(kept_bits) != b).sum(1)
        j = int(d.argmin())
        if d[j] <= HAMMING:
            dup_of[ok.loc[i, 'file']] = ok.loc[keep[j], 'file']; continue
    kept_bits.append(b); keep.append(i)

clean = ok.loc[keep].drop(columns=['bits']).reset_index(drop=True)
clean.to_csv(OUT / 'candidates.csv', index=False)
print(f'Near-duplicates removed: {len(dup_of)}  |  candidates for labelling: {len(clean)}')
clean.groupby('bucket').size()

In [ ]:
# 6. CLIP pre-screen: a zero-shot SUGGESTION for each image (humans make the final call)
import torch, open_clip
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
clip_model, _, clip_pre = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
clip_model = clip_model.to(dev).eval(); tok = open_clip.get_tokenizer('ViT-B-32')

PROMPTS = {
    'plastic_clean': ['a photo of empty plastic bottles collected for recycling', 'a photo of clean plastic water sachets',
                      'a pile of sorted plastic waste', 'a bale of compressed plastic bottles', 'a sack full of plastic bottles'],
    'plastic_contaminated': ['a pile of mixed garbage with plastic and food waste', 'dirty plastic waste mixed with rubbish',
                             'an overflowing rubbish bin', 'plastic waste in a dirty gutter', 'a rubbish dump with plastic bags'],
    'non_plastic': ['a pile of scrap metal and cans', 'a pile of glass bottles', 'a pile of food and vegetable waste',
                    'a pile of old clothes and textiles', 'a pile of cardboard and paper waste'],
    'discard_junk': ['a poster or infographic with text', 'a logo', 'a map', 'a cartoon illustration', 'a screenshot',
                     'a photo of a building or street with no rubbish', 'a collage of several photos'],
    'discard_person': ['a close-up portrait of a person', 'a group of people posing for a photo'],
}
with torch.no_grad():
    T = []
    for k in PROMPTS:
        e = clip_model.encode_text(tok(PROMPTS[k]).to(dev)); e = e / e.norm(dim=-1, keepdim=True)
        m = e.mean(0); T.append(m / m.norm())
    T = torch.stack(T)

sugg, conf = [], []
files = list(clean['file'])
for i in range(0, len(files), 64):
    x = torch.stack([clip_pre(Image.open(IMG_DIR / f).convert('RGB')) for f in files[i:i + 64]]).to(dev)
    with torch.no_grad():
        v = clip_model.encode_image(x); v = v / v.norm(dim=-1, keepdim=True)
        p = (100 * v @ T.T).softmax(-1).cpu().numpy()
    keys = list(PROMPTS); sugg += [keys[j] for j in p.argmax(1)]; conf += list(p.max(1))
clean['suggestion'], clean['suggestion_conf'] = sugg, np.round(conf, 3)
clean.to_csv(OUT / 'candidates.csv', index=False)
print(clean['suggestion'].value_counts())

In [ ]:
# 7. Split the labelling work across the team.
# The first N_OVERLAP images go to EVERYONE, so we can measure agreement (Cohen's kappa) for the report.
TEAM = ['nana', 'kelvin', 'cajetan']
N_OVERLAP = 100
cand = pd.read_csv(OUT / 'candidates.csv').sample(frac=1, random_state=42).reset_index(drop=True)
cand['assigned'] = ''
cand.loc[:N_OVERLAP - 1, 'assigned'] = 'ALL'
rest = cand.index[N_OVERLAP:]
cand.loc[rest, 'assigned'] = [TEAM[i % len(TEAM)] for i in range(len(rest))]
cand.to_csv(OUT / 'assignments.csv', index=False)
print(cand['assigned'].value_counts())
print(f'Each person labels about {N_OVERLAP + len(rest) // len(TEAM)} images (~2-3 s each).')

In [ ]:
# 8. LABELLING TOOL — each teammate runs cells 1, 2 and this cell, with their own name.
LABELER = 'nana'      # <-- change to your name: 'nana', 'kelvin' or 'cajetan'

import ipywidgets as W
from IPython.display import display
cand = pd.read_csv(OUT / 'assignments.csv')
mine = cand[cand['assigned'].isin(['ALL', LABELER])].reset_index(drop=True)
LAB_FILE = OUT / f'labels_{LABELER}.csv'
labels = pd.read_csv(LAB_FILE).drop_duplicates('file', keep='last').set_index('file')['label'].to_dict() \
         if LAB_FILE.exists() else {}
order = list(mine['file']); info = mine.set_index('file')
pos = next((i for i, f in enumerate(order) if f not in labels), len(order))

img = W.Image(format='jpeg', width=560)
meta, prog = W.HTML(), W.HTML()
BTN = {'plastic_clean': ('1 · plastic clean', 'success'), 'plastic_contaminated': ('2 · plastic contaminated', 'warning'),
       'non_plastic': ('3 · non-plastic', 'info'), 'discard_junk': ('discard: junk / no waste', ''),
       'discard_person': ('discard: identifiable person', 'danger')}
buttons = {k: W.Button(description=d, button_style=s, layout=W.Layout(width='230px')) for k, (d, s) in BTN.items()}
back = W.Button(description='◀ back', layout=W.Layout(width='100px'))

def show():
    n_done = sum(f in labels for f in order)
    prog.value = f'<b>{LABELER}</b>: {n_done}/{len(order)} labelled'
    if pos >= len(order):
        img.value = b''; meta.value = '<h3>All done — thank you!</h3>'; return
    f = order[pos]; r = info.loc[f]
    img.value = (IMG_DIR / f).read_bytes()
    cur = labels.get(f, '—')
    meta.value = (f"#{pos + 1} &nbsp; query: <i>{r['query']}</i> &nbsp;|&nbsp; "
                  f"CLIP hint: {r['suggestion']} ({r['suggestion_conf']:.2f}) &nbsp;|&nbsp; current: <b>{cur}</b>")

def on_label(k):
    def h(_):
        global pos
        if pos >= len(order): return
        f = order[pos]; labels[f] = k
        pd.DataFrame([{'file': f, 'label': k, 'labeler': LABELER, 'ts': time.time()}]) \
          .to_csv(LAB_FILE, mode='a', header=not LAB_FILE.exists(), index=False)
        pos += 1; show()
    return h

def on_back(_):
    global pos
    pos = max(0, pos - 1); show()

for k, b in buttons.items(): b.on_click(on_label(k))
back.on_click(on_back)
display(W.VBox([prog, img, meta,
                W.HBox([buttons['plastic_clean'], buttons['plastic_contaminated'], buttons['non_plastic']]),
                W.HBox([buttons['discard_junk'], buttons['discard_person'], back])]))
show()

In [ ]:
# 9. Agreement on the shared overlap set (run after everyone finishes)
from sklearn.metrics import cohen_kappa_score
from itertools import combinations
cand = pd.read_csv(OUT / 'assignments.csv')
lab = {p: pd.read_csv(OUT / f'labels_{p}.csv').drop_duplicates('file', keep='last').set_index('file')['label']
       for p in TEAM if (OUT / f'labels_{p}.csv').exists()}
if not lab:
    raise SystemExit('No labels yet. Label images in cell 8 first (click the buttons), then rerun this cell.')
overlap = cand.loc[cand['assigned'] == 'ALL', 'file']
for a, b in combinations(lab, 2):
    common = [f for f in overlap if f in lab[a].index and f in lab[b].index]
    if common:
        k = cohen_kappa_score(lab[a][common], lab[b][common])
        agree = np.mean(lab[a][common].values == lab[b][common].values)
        print(f'{a} vs {b}: {len(common)} shared images | raw agreement {agree:.1%} | Cohen kappa {k:.2f}')
print('\nKappa > 0.6 = substantial agreement. If lower, discuss the disagreements and refine the rules before using the data.')

In [ ]:
# 10. Export the final dataset for fine-tuning
import shutil
DATA = OUT / 'dataset'
lab_files = [OUT / f'labels_{p}.csv' for p in TEAM if (OUT / f'labels_{p}.csv').exists()]
if not lab_files:
    raise SystemExit('No labels yet. Label images in cell 8 first (click the buttons), then rerun this cell.')
lab = {f.stem.replace('labels_', ''): f for f in lab_files}
all_lab = pd.concat([pd.read_csv(OUT / f'labels_{p}.csv') for p in lab]).sort_values('ts')
all_lab = all_lab.drop_duplicates(['file', 'labeler'], keep='last')
# overlap images: majority vote; ties -> discard (too ambiguous to train on)
def resolve(labels):
    vc = labels.value_counts()
    return vc.index[0] if (len(vc) == 1 or vc.iloc[0] > vc.iloc[1]) else 'discard_tie'
final = all_lab.groupby('file')['label'].apply(resolve).rename('label').reset_index()
final = final.merge(pd.read_csv(OUT / 'candidates.csv'), on='file', how='left')
keep = final[final['label'].isin(CLASSES)]

for c in CLASSES: (DATA / c).mkdir(parents=True, exist_ok=True)
for _, r in keep.iterrows():
    dst = DATA / r['label'] / r['file']
    if not dst.exists(): shutil.copy(IMG_DIR / r['file'], dst)
keep[['file', 'label', 'bucket', 'query', 'image_url', 'source_page', 'title']].to_csv(OUT / 'dataset_manifest.csv', index=False)

print(final['label'].value_counts(), '\n')
print('Dataset written to', DATA)
print('Next: the Stage 2 fine-tuning notebook reads dataset/ and dataset_manifest.csv.')